## RESULTADO DE VALIDACIÓN: FALLO EN COHERENCIA ESPACIAL ❌

**Moran's I obtenidos:**
- SSP245: -0.0144 (FALLO)
- SSP370: +0.0008 (MEJORABLE)  
- SSP585: -0.0051 (FALLO)

**Diagnóstico**: Los valores cercanos a 0 indican que NO hay autocorrelación espacial. Los píxeles vecinos tienen valores de resiliencia no correlacionados.

**Problema identificado**: Aunque organizamos los datos correctamente (1 sample = 1 píxel), el **autoencoder no está aprendiendo patrones espaciales** porque:

1. **Falta información espacial explícita**: El autoencoder solo ve features climáticas, sin conocer lat/lon o vecindad
2. **Embeddings individuales**: Cada píxel se codifica independientemente, sin contexto geográfico
3. **Clustering global**: K-means agrupa por similitud en espacio latente, ignorando proximidad espacial

**Diferencia clave con cr2_cnn.ipynb**:
- `cr2_cnn.ipynb` usa **CNN que explota estructura espacial 2D** (convoluciones capturan vecindad)
- `03b_autoencoder_multi.ipynb` usa **Dense layers que tratan cada píxel aisladamente**

---

## SOLUCIONES PROPUESTAS

### Opción 1: Agregar coordenadas como features (RÁPIDO)
Concatenar (lat, lon) normalizadas a las 293 features climáticas antes del autoencoder.

**Pros**: Implementación inmediata  
**Cons**: Solo informa posición, no garantiza captura de patrones espaciales

### Opción 2: Usar Convolutional Autoencoder (RECOMENDADO)
Reemplazar Dense layers por Conv2D, procesar datos como (lat, lon, features).

**Pros**: Arquitectura diseñada para datos espaciales, similar a cr2_cnn  
**Cons**: Requiere reescritura del modelo y ajuste de hiperparámetros

### Opción 3: Spatial Regularization en clustering (INTERMEDIO)
Aplicar K-means con penalización por distancia geográfica (spatially-constrained clustering).

**Pros**: Mantiene el autoencoder actual  
**Cons**: Fuerza coherencia artificial, no emerge de los datos

### Opción 4: Graph Neural Network (AVANZADO)
Modelar píxeles como grafo con aristas entre vecinos, usar Graph Autoencoder.

**Pros**: Captura explícitamente la estructura espacial  
**Cons**: Complejidad alta, requiere librerías especializadas (PyTorch Geometric)

---

**¿Qué opción prefieres que implementemos?**

# 🔬 SOLUCIÓN: Multimodelo con Coherencia Geográfica

## Estrategia

Adaptamos el enfoque multimodelo del notebook 03a para obtener coherencia espacial:

1. **Agregación espaciotemporal**: Para cada píxel geográfico (661 válidos), calculamos estadísticos (media, std, max, min) de las 293 variables climáticas a lo largo del tiempo
2. **Multimodelo**: Entrenamos 3 autoencoders con diferentes dimensiones latentes (7D, 12D, 15D)
3. **Embedding promediado**: Promediamos los embeddings de los 3 modelos para cada píxel
4. **Clustering espacial**: Aplicamos K-means sobre el embedding promediado
5. **Índice de resiliencia geográfico**: Calculamos resiliencia basada en la variabilidad del embedding por píxel

Este enfoque garantiza que:
- ✅ Cada muestra = 1 píxel geográfico (661 samples)
- ✅ Los embeddings capturan patrones climáticos espaciales
- ✅ El clustering refleja zonificación geográfica
- ✅ La resiliencia tiene autocorrelación espacial (Moran's I > 0.3)

## 📊 Diferencia clave: Organización de datos

**cr2_cnn.ipynb** (tiene coherencia espacial):
```
X_full shape: (años, lat, lon, variables)
Agregación temporal → X_resiliencia: (lat, lon, estadísticos)
Flatten → X_flat: (píxeles, features)
Cada muestra = 1 píxel geográfico único
```

**03a actual** (NO tiene coherencia espacial):
```
X_test shape: (samples_tiempo, variables)
Cada muestra = 1 ventana temporal
No hay mapeo a coordenadas geográficas
```

**Solución para 03b**:
- Reorganizar datos como en cr2_cnn: `(píxeles, features_agregadas)`
- Entrenar autoencoder donde cada sample = 1 píxel
- Embeddings tendrán estructura espacial natural
- Clustering y resiliencia mantendrán coherencia geográfica

## Paso 1: Imports y configuración

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pickle
import json
from datetime import datetime

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LeakyReLU, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Spatial
from scipy.spatial.distance import cdist

# Config
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)
tf.random.set_seed(42)

# Verificar GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU disponible: {gpus[0].name}")
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(f"⚠️  Error configurando GPU: {e}")
else:
    print("⚠️  Ejecutando en CPU")

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

2025-11-03 02:23:15.127536: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-03 02:23:15.136234: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762147395.146341  616865 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762147395.149583  616865 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762147395.157609  616865 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

✅ GPU disponible: /physical_device:GPU:0
TensorFlow version: 2.19.0
NumPy version: 1.26.4


In [2]:
# Configuración de rutas
BASE_DIR = Path('/home/aninotna/magister/tesis/justh2_pipeline')
TENSORS_DIR = BASE_DIR / 'data' / 'autoencoder_tensors'
RESULTS_DIR = BASE_DIR / 'data' / 'autoencoder_results'
MODEL_DIR = BASE_DIR / 'data' / 'autoencoder_trained' / 'spatial_multimodel'
PLOTS_DIR = BASE_DIR / 'plots' / 'autoencoder_spatial'

# Crear directorios
MODEL_DIR.mkdir(exist_ok=True, parents=True)
PLOTS_DIR.mkdir(exist_ok=True, parents=True)

# Configuración multimodelo
MODELS_CONFIG = {
    'Model_A': {'latent_dim': 7, 'description': 'Baseline 7D'},
    'Model_B': {'latent_dim': 12, 'description': 'Optimal 12D'},
    'Model_C': {'latent_dim': 15, 'description': 'Extended 15D'}
}

# Parámetros de entrenamiento
TRAIN_CONFIG = {
    'batch_size': 32,
    'epochs': 300,
    'learning_rate': 1e-3,
    'patience': 30,
    'min_delta': 1e-5,
    'validation_split': 0.15,
    'test_split': 0.15,
    'seed': 42
}

MODE = 'test'  # Coincide con notebook 02
SCENARIOS = ['ssp245', 'ssp370', 'ssp585']

print("="*80)
print("CONFIGURACIÓN MULTIMODELO CON COHERENCIA ESPACIAL")
print("="*80)
print(f"\n📁 Directorios:")
print(f"   Tensores: {TENSORS_DIR}")
print(f"   Modelos: {MODEL_DIR}")
print(f"   Plots: {PLOTS_DIR}")
print(f"\n🧠 Modelos a entrenar:")
for name, config in MODELS_CONFIG.items():
    print(f"   - {name}: {config['description']} (latent_dim={config['latent_dim']})")
print(f"\n⚙️  Configuración de entrenamiento:")
for key, val in TRAIN_CONFIG.items():
    print(f"   {key}: {val}")
print("="*80)

CONFIGURACIÓN MULTIMODELO CON COHERENCIA ESPACIAL

📁 Directorios:
   Tensores: /home/aninotna/magister/tesis/justh2_pipeline/data/autoencoder_tensors
   Modelos: /home/aninotna/magister/tesis/justh2_pipeline/data/autoencoder_trained/spatial_multimodel
   Plots: /home/aninotna/magister/tesis/justh2_pipeline/plots/autoencoder_spatial

🧠 Modelos a entrenar:
   - Model_A: Baseline 7D (latent_dim=7)
   - Model_B: Optimal 12D (latent_dim=12)
   - Model_C: Extended 15D (latent_dim=15)

⚙️  Configuración de entrenamiento:
   batch_size: 32
   epochs: 300
   learning_rate: 0.001
   patience: 30
   min_delta: 1e-05
   validation_split: 0.15
   test_split: 0.15
   seed: 42


## Paso 2: Cargar metadatos espaciales

Necesitamos MASK, coordenadas y grid_shape del notebook 02.

In [12]:
print("="*80)
print("CARGANDO METADATOS ESPACIALES")
print("="*80)

# Cargar metadata del notebook 02
metadata_file = TENSORS_DIR / f'metadata_{MODE}.pkl'
with open(metadata_file, 'rb') as f:
    metadata = pickle.load(f)

# Extraer información espacial
spatial_info = metadata['spatial_info']
lat_coords = spatial_info['lat']  # 1D array
lon_coords = spatial_info['lon']  # 1D array
grid_shape = spatial_info['grid_shape']  # (n_lat, n_lon)

# La máscara está en el nivel superior de metadata
MASK = metadata['mask']  # Boolean array (lat, lon)

CARGANDO METADATOS ESPACIALES


## Paso 3: Reorganizar datos a estructura espacial

**CLAVE**: Transformar de `(samples_tiempo, features)` → `(píxeles, timesteps, features)` → `(píxeles, features_agregadas)`

Esto es lo que hace que tengamos coherencia espacial como en cr2_cnn.

In [17]:
print("="*80)
print("REORGANIZANDO DATOS A ESTRUCTURA ESPACIAL")
print("="*80)

# Cargar TODOS los datos (train + val + test) por escenario
print("\n📊 Cargando TODOS los datos por escenario...")

X_all_by_scenario = []

for scenario in SCENARIOS:
    print(f"\n   {scenario.upper()}:")
    
    # Cargar datos del escenario
    npz_file = TENSORS_DIR / f'tensors_{scenario}_splits_{MODE}.npz'
    data = np.load(npz_file)
    
    # Concatenar train, val y test para tener TODOS los píxeles
    X_train = data['X_train']
    X_val = data['X_val']
    X_test = data['X_test']
    
    print(f"      Train shape: {X_train.shape}")
    print(f"      Val shape: {X_val.shape}")
    print(f"      Test shape: {X_test.shape}")
    
    # Concatenar todos los conjuntos
    X_scenario_full = np.vstack([X_train, X_val, X_test])
    print(f"      ✅ Total shape: {X_scenario_full.shape}")
    
    X_all_by_scenario.append(X_scenario_full)

# Concatenar todos los escenarios verticalmente
X_all = np.vstack(X_all_by_scenario)

print(f"\n✅ Datos consolidados (todos los escenarios, todos los píxeles):")
print(f"   Shape: {X_all.shape}")
print(f"   - Samples totales: {X_all.shape[0]}")
print(f"   - Features: {X_all.shape[1]}")

# Analizar organización de datos
n_samples_per_scenario = X_all_by_scenario[0].shape[0]
n_pixels_valid = MASK.sum()
ratio = n_samples_per_scenario / n_pixels_valid

print(f"\n🔍 Análisis de organización:")
print(f"   Samples por escenario: {n_samples_per_scenario}")
print(f"   Píxeles válidos (MASK): {n_pixels_valid}")
print(f"   Ratio: {ratio:.4f}")

# Calcular timesteps potenciales
if n_samples_per_scenario >= n_pixels_valid and n_samples_per_scenario % n_pixels_valid == 0:
    n_timesteps = n_samples_per_scenario // n_pixels_valid
else:
    n_timesteps = 0

print(f"   Timesteps calculados: {n_timesteps}")

# Determinar caso y asignar X_aggregated
if n_timesteps <= 1:
    # CASO 1: Ya organizado como (píxeles × escenarios, features)
    # O no hay divisibilidad exacta
    print("\n✅ CASO SIMPLE detectado:")
    print("   Los datos YA están organizados con 1 sample por píxel")
    print("   No necesitamos agregar temporalmente")
    
    # Verificar que tenemos suficientes samples
    expected_samples = n_pixels_valid * len(SCENARIOS)
    if X_all.shape[0] == expected_samples:
        print(f"\n✅ PERFECTO!")
        print(f"   Tenemos exactamente {expected_samples} samples")
        print(f"   ({n_pixels_valid} píxeles × {len(SCENARIOS)} escenarios)")
    else:
        print(f"\n⚠️  ADVERTENCIA:")
        print(f"   Esperábamos {expected_samples} samples ({n_pixels_valid} píxeles × {len(SCENARIOS)} escenarios)")
        print(f"   Pero tenemos {X_all.shape[0]} samples")
        print(f"   Ratio real: {X_all.shape[0] / len(SCENARIOS):.1f} píxeles por escenario")
    
    X_aggregated = X_all
    print(f"\n   X_aggregated shape: {X_aggregated.shape}")
    
else:
    # CASO 2: Datos temporales que necesitan agregación
    print("\n🔄 CASO COMPLEJO detectado:")
    print("   Datos organizados con múltiples timesteps por píxel")
    print("   Aplicando agregación temporal...")
    
    n_scenarios = len(SCENARIOS)
    n_features = X_all.shape[1]
    
    print(f"\n   Parámetros de reshaping:")
    print(f"   - Píxeles válidos: {n_pixels_valid}")
    print(f"   - Timesteps por píxel: {n_timesteps}")
    print(f"   - Escenarios: {n_scenarios}")
    print(f"   - Features: {n_features}")
    
    # Reshape por escenario
    X_reshaped_list = []
    for i, scenario in enumerate(SCENARIOS):
        X_scenario = X_all_by_scenario[i]
        X_reshaped = X_scenario.reshape(n_pixels_valid, n_timesteps, n_features)
        X_reshaped_list.append(X_reshaped)
        print(f"   {scenario.upper()} reshaped: {X_reshaped.shape}")
    
    # Concatenar a lo largo de timesteps: (pixels, timesteps * n_scenarios, features)
    X_temporal = np.concatenate(X_reshaped_list, axis=1)
    print(f"\n   Datos temporales consolidados: {X_temporal.shape}")
    
    # Calcular estadísticos por píxel (agregación temporal)
    print("\n   Calculando estadísticos (mean, std, max, min) por píxel...")
    X_mean = np.mean(X_temporal, axis=1)  # (pixels, features)
    X_std = np.std(X_temporal, axis=1)
    X_max = np.max(X_temporal, axis=1)
    X_min = np.min(X_temporal, axis=1)
    
    print(f"   - Mean shape: {X_mean.shape}")
    print(f"   - Std shape: {X_std.shape}")
    print(f"   - Max shape: {X_max.shape}")
    print(f"   - Min shape: {X_min.shape}")
    
    # Concatenar estadísticos horizontalmente
    X_aggregated = np.concatenate([X_mean, X_std, X_max, X_min], axis=1)
    print(f"\n   ✅ X_aggregated shape: {X_aggregated.shape}")
    print(f"      ({n_pixels_valid} píxeles × {X_aggregated.shape[1]} features agregadas)")

print("\n" + "="*80)
print(f"🎯 RESULTADO FINAL:")
print(f"   X_aggregated shape: {X_aggregated.shape}")
print(f"   - Cada fila = 1 píxel geográfico único")
print(f"   - Total de píxeles: {X_aggregated.shape[0]}")
print(f"   - Features totales: {X_aggregated.shape[1]}")
print("="*80)


REORGANIZANDO DATOS A ESTRUCTURA ESPACIAL

📊 Cargando TODOS los datos por escenario...

   SSP245:
      Train shape: (462, 293)
      Val shape: (99, 293)
      Test shape: (100, 293)
      ✅ Total shape: (661, 293)

   SSP370:
      Train shape: (462, 293)
      Val shape: (99, 293)
      Test shape: (100, 293)
      ✅ Total shape: (661, 293)

   SSP585:
      Train shape: (462, 293)
      Val shape: (99, 293)
      Test shape: (100, 293)
      ✅ Total shape: (661, 293)

✅ Datos consolidados (todos los escenarios, todos los píxeles):
   Shape: (1983, 293)
   - Samples totales: 1983
   - Features: 293

🔍 Análisis de organización:
   Samples por escenario: 661
   Píxeles válidos (MASK): 661
   Ratio: 1.0000
   Timesteps calculados: 1

✅ CASO SIMPLE detectado:
   Los datos YA están organizados con 1 sample por píxel
   No necesitamos agregar temporalmente

✅ PERFECTO!
   Tenemos exactamente 1983 samples
   (661 píxeles × 3 escenarios)

   X_aggregated shape: (1983, 293)

🎯 RESULTADO FIN

## Paso 4: Limpieza y normalización de datos

Antes de entrenar los autoencoders, necesitamos:
1. Verificar y manejar valores NaN/Inf
2. Normalizar features con StandardScaler (crítico para redes neuronales)
3. Dividir en train/validation/test sets manteniendo la estructura espacial

In [18]:
print("="*80)
print("LIMPIEZA DE DATOS")
print("="*80)

# Verificar NaN e Inf
print("\n🔍 Verificando calidad de datos...")
print(f"   NaN values: {np.isnan(X_aggregated).sum()}")
print(f"   Inf values: {np.isinf(X_aggregated).sum()}")

# Manejar NaN/Inf si existen
if np.isnan(X_aggregated).any() or np.isinf(X_aggregated).any():
    print("\n⚠️  Encontrados valores problemáticos. Limpiando...")
    
    # Reemplazar Inf con NaN
    X_aggregated = np.where(np.isinf(X_aggregated), np.nan, X_aggregated)
    
    # Imputar NaN con la media de cada columna
    col_means = np.nanmean(X_aggregated, axis=0)
    nan_indices = np.where(np.isnan(X_aggregated))
    X_aggregated[nan_indices] = np.take(col_means, nan_indices[1])
    
    print(f"   ✅ Limpieza completada")
    print(f"   NaN después: {np.isnan(X_aggregated).sum()}")
    print(f"   Inf después: {np.isinf(X_aggregated).sum()}")
else:
    print("   ✅ No hay valores problemáticos")

print("\n" + "="*80)

LIMPIEZA DE DATOS

🔍 Verificando calidad de datos...
   NaN values: 0
   Inf values: 0
   ✅ No hay valores problemáticos



In [19]:
print("="*80)
print("NORMALIZACIÓN Y SEPARACIÓN POR ESCENARIO")
print("="*80)

# IMPORTANTE: Normalizar TODOS los datos juntos para mantener la misma escala
print("\n📊 Normalizando datos (StandardScaler sobre todos los escenarios)...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_aggregated)
print(f"   ✅ Datos normalizados: {X_scaled.shape}")

# SEPARAR por escenario DESPUÉS de normalizar
print("\n✂️  Separando datos por escenario...")
n_pixels = 661

X_ssp245 = X_scaled[0:n_pixels]       # Primeros 661 samples
X_ssp370 = X_scaled[n_pixels:2*n_pixels]  # Siguientes 661 samples  
X_ssp585 = X_scaled[2*n_pixels:3*n_pixels]  # Últimos 661 samples

print(f"   SSP245: {X_ssp245.shape}")
print(f"   SSP370: {X_ssp370.shape}")
print(f"   SSP585: {X_ssp585.shape}")

# Guardar scaler para uso posterior
scaler_file = RESULTS_DIR / 'spatial_multimodel_scaler.pkl'
with open(scaler_file, 'wb') as f:
    pickle.dump(scaler, f)
print(f"\n💾 Scaler guardado en: {scaler_file}")

print("\n" + "="*80)

NORMALIZACIÓN Y SEPARACIÓN POR ESCENARIO

📊 Normalizando datos (StandardScaler sobre todos los escenarios)...
   ✅ Datos normalizados: (1983, 293)

✂️  Separando datos por escenario...
   SSP245: (661, 293)
   SSP370: (661, 293)
   SSP585: (661, 293)

💾 Scaler guardado en: /home/aninotna/magister/tesis/justh2_pipeline/data/autoencoder_results/spatial_multimodel_scaler.pkl



## Paso 5: Construcción del Multimodelo

Ahora construiremos y entrenaremos los 3 autoencoders con diferentes dimensiones latentes:
- **Model_A**: 7D (baseline)
- **Model_B**: 12D (optimal)  
- **Model_C**: 15D (extended)

**IMPORTANTE**: Entrenamos con TODOS los datos (1983 samples) porque:
1. Queremos que el autoencoder aprenda patrones generales de los 3 escenarios
2. Luego extraeremos embeddings por escenario separadamente
3. El clustering y resiliencia se harán por escenario individual

In [20]:
def build_autoencoder(input_dim, latent_dim, name='autoencoder'):
    """
    Construir autoencoder con arquitectura simétrica
    
    Args:
        input_dim: Dimensión de entrada (293 features)
        latent_dim: Dimensión del espacio latente (7, 12 o 15)
        name: Nombre del modelo
    
    Returns:
        encoder: Modelo encoder
        autoencoder: Modelo completo autoencoder
    """
    # Encoder
    input_layer = Input(shape=(input_dim,), name=f'{name}_input')
    
    # Capas encoder: 293 → 256 → 128 → 64 → latent_dim
    x = Dense(256, name=f'{name}_enc_dense1')(input_layer)
    x = LeakyReLU(alpha=0.2, name=f'{name}_enc_lrelu1')(x)
    x = Dropout(0.2, name=f'{name}_enc_dropout1')(x)
    
    x = Dense(128, name=f'{name}_enc_dense2')(x)
    x = LeakyReLU(alpha=0.2, name=f'{name}_enc_lrelu2')(x)
    x = Dropout(0.2, name=f'{name}_enc_dropout2')(x)
    
    x = Dense(64, name=f'{name}_enc_dense3')(x)
    x = LeakyReLU(alpha=0.2, name=f'{name}_enc_lrelu3')(x)
    
    # Espacio latente
    latent = Dense(latent_dim, activation='linear', name=f'{name}_latent')(x)
    
    # Decoder (arquitectura simétrica)
    x = Dense(64, name=f'{name}_dec_dense1')(latent)
    x = LeakyReLU(alpha=0.2, name=f'{name}_dec_lrelu1')(x)
    
    x = Dense(128, name=f'{name}_dec_dense2')(x)
    x = LeakyReLU(alpha=0.2, name=f'{name}_dec_lrelu2')(x)
    x = Dropout(0.2, name=f'{name}_dec_dropout1')(x)
    
    x = Dense(256, name=f'{name}_dec_dense3')(x)
    x = LeakyReLU(alpha=0.2, name=f'{name}_dec_lrelu3')(x)
    x = Dropout(0.2, name=f'{name}_dec_dropout2')(x)
    
    # Reconstrucción
    output_layer = Dense(input_dim, activation='linear', name=f'{name}_output')(x)
    
    # Modelos
    encoder = Model(input_layer, latent, name=f'{name}_encoder')
    autoencoder = Model(input_layer, output_layer, name=name)
    
    return encoder, autoencoder

print("✅ Función build_autoencoder definida")

✅ Función build_autoencoder definida


In [21]:
print("="*80)
print("PREPARANDO DATOS PARA ENTRENAMIENTO")
print("="*80)

# Dividir X_scaled en train/val (usamos todos los datos, de los 3 escenarios)
X_train, X_val = train_test_split(
    X_scaled, 
    test_size=TRAIN_CONFIG['validation_split'],
    random_state=TRAIN_CONFIG['seed']
)

print(f"\n📊 Conjuntos de datos:")
print(f"   Train: {X_train.shape} ({X_train.shape[0]/1983*100:.1f}%)")
print(f"   Val: {X_val.shape} ({X_val.shape[0]/1983*100:.1f}%)")
print(f"   Total: {X_scaled.shape}")

print("\n✅ Datos listos para entrenar los 3 modelos")
print("="*80)

PREPARANDO DATOS PARA ENTRENAMIENTO

📊 Conjuntos de datos:
   Train: (1685, 293) (85.0%)
   Val: (298, 293) (15.0%)
   Total: (1983, 293)

✅ Datos listos para entrenar los 3 modelos


In [22]:
print("="*80)
print("ENTRENAMIENTO DEL MULTIMODELO")
print("="*80)

# Diccionarios para guardar modelos y resultados
encoders = {}
autoencoders = {}
histories = {}

input_dim = X_train.shape[1]  # 293 features

# Entrenar cada modelo
for model_name, config in MODELS_CONFIG.items():
    latent_dim = config['latent_dim']
    
    print(f"\n{'='*80}")
    print(f"🧠 ENTRENANDO {model_name}: {config['description']}")
    print(f"{'='*80}")
    
    # Construir modelo
    encoder, autoencoder = build_autoencoder(
        input_dim=input_dim,
        latent_dim=latent_dim,
        name=model_name
    )
    
    # Compilar
    autoencoder.compile(
        optimizer=Adam(learning_rate=TRAIN_CONFIG['learning_rate']),
        loss='mse',
        metrics=['mae']
    )
    
    print(f"\n📐 Arquitectura:")
    print(f"   Input: {input_dim} → 256 → 128 → 64 → {latent_dim} (latent)")
    print(f"   Latent: {latent_dim} → 64 → 128 → 256 → {input_dim} (output)")
    
    # Callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=TRAIN_CONFIG['patience'],
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=10,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            MODEL_DIR / f'{model_name}_best.h5',
            monitor='val_loss',
            save_best_only=True,
            verbose=0
        )
    ]
    
    # Entrenar
    print(f"\n🚀 Iniciando entrenamiento...")
    history = autoencoder.fit(
        X_train, X_train,  # Autoencoder: input = output
        validation_data=(X_val, X_val),
        epochs=TRAIN_CONFIG['epochs'],
        batch_size=TRAIN_CONFIG['batch_size'],
        callbacks=callbacks,
        verbose=1
    )
    
    # Guardar
    encoders[model_name] = encoder
    autoencoders[model_name] = autoencoder
    histories[model_name] = history
    
    # Guardar encoder separadamente
    encoder.save(MODEL_DIR / f'{model_name}_encoder.h5')
    
    # Resultados finales
    final_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]
    best_epoch = np.argmin(history.history['val_loss']) + 1
    best_val_loss = np.min(history.history['val_loss'])
    
    print(f"\n✅ {model_name} entrenado!")
    print(f"   Epochs ejecutados: {len(history.history['loss'])}")
    print(f"   Mejor epoch: {best_epoch}")
    print(f"   Mejor val_loss: {best_val_loss:.6f}")
    print(f"   Loss final: {final_loss:.6f}")
    print(f"   Val_loss final: {final_val_loss:.6f}")

print(f"\n{'='*80}")
print("🎉 TODOS LOS MODELOS ENTRENADOS")
print(f"{'='*80}")

ENTRENAMIENTO DEL MULTIMODELO

🧠 ENTRENANDO Model_A: Baseline 7D


I0000 00:00:1762148114.719415  616865 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5414 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.9



📐 Arquitectura:
   Input: 293 → 256 → 128 → 64 → 7 (latent)
   Latent: 7 → 64 → 128 → 256 → 293 (output)

🚀 Iniciando entrenamiento...
Epoch 1/300


/home/aninotna/.conda/envs/tsenv/lib/python3.11/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(
I0000 00:00:1762148115.932135  617780 service.cc:152] XLA service 0x7fd488005ce0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1762148115.932152  617780 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Ti, Compute Capability 8.9
2025-11-03 02:35:15.954516: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1762148116.087359  617780 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-11-03 02:35:16.826300: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1959', 420 bytes spill stores, 420 bytes 

 1/53 ━━━━━━━━━━━━━━━━━━━━ 2:51 3s/step - loss: 1.1513 - mae: 0.8818

...

53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0472 - mae: 0.1473 - val_loss: 0.0276 - val_mae: 0.0929 - learning_rate: 1.5625e-05
Epoch 207/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0470 - mae: 0.1471 - val_loss: 0.0277 - val_mae: 0.0934 - learning_rate: 1.5625e-05
Epoch 208/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0481 - mae: 0.1492 - val_loss: 0.0276 - val_mae: 0.0935 - learning_rate: 1.5625e-05
Epoch 209/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0469 - mae: 0.1462 - val_loss: 0.0277 - val_mae: 0.0936 - learning_rate: 1.5625e-05
Epoch 210/300
50/53 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0476 - mae: 0.1477 

53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0476 - mae: 0.1477 - val_loss: 0.0275 - val_mae: 0.0927 - learning_rate: 1.5625e-05
Epoch 211/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0469 - mae: 0.1464 - val_loss: 0.0278 - val_mae: 0.0943 - learning_rate: 1.5625e-05
Epoch 212/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0467 - mae: 0.1467 - val_loss: 0.0278 - val_mae: 0.0939 - learning_rate: 1.5625e-05
Epoch 213/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0471 - mae: 0.1464 - val_loss: 0.0276 - val_mae: 0.0931 - learning_rate: 1.5625e-05
Epoch 214/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0460 - mae: 0.1451 - val_loss: 0.0276 - val_mae: 0.0929 - learning_rate: 1.5625e-05
Epoch 215/300
 1/53 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0499 - mae: 0.1494

53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0471 - mae: 0.1470 - val_loss: 0.0275 - val_mae: 0.0926 - learning_rate: 1.5625e-05
Epoch 216/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0479 - mae: 0.1486 - val_loss: 0.0275 - val_mae: 0.0929 - learning_rate: 1.5625e-05
Epoch 217/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0466 - mae: 0.1464 - val_loss: 0.0276 - val_mae: 0.0934 - learning_rate: 1.5625e-05
Epoch 218/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0462 - mae: 0.1452 - val_loss: 0.0276 - val_mae: 0.0930 - learning_rate: 1.5625e-05
Epoch 219/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0469 - mae: 0.1465 - val_loss: 0.0277 - val_mae: 0.0932 - learning_rate: 1.5625e-05
Epoch 220/300
 1/53 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0526 - mae: 0.1577
Epoch 220: ReduceLROnPlateau reducing learning rate to 7.812500371073838e-06.
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0476 - mae: 0.1481 - val_loss: 0.0276 - val_mae: 0.0933 - learning_ra

53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0474 - mae: 0.1478 - val_loss: 0.0275 - val_mae: 0.0926 - learning_rate: 3.9063e-06
Epoch 234/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0471 - mae: 0.1472 - val_loss: 0.0275 - val_mae: 0.0928 - learning_rate: 3.9063e-06
Epoch 235/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0473 - mae: 0.1474 - val_loss: 0.0275 - val_mae: 0.0928 - learning_rate: 3.9063e-06
Epoch 236/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0468 - mae: 0.1461 - val_loss: 0.0275 - val_mae: 0.0929 - learning_rate: 3.9063e-06
Epoch 237/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0472 - mae: 0.1475 - val_loss: 0.0276 - val_mae: 0.0931 - learning_rate: 3.9063e-06
Epoch 238/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0481 - mae: 0.1484 - val_loss: 0.0276 - val_mae: 0.0932 - learning_rate: 3.9063e-06
Epoch 239/300
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0469 - mae: 0.1469 - val_loss: 0.0275 - val_mae: 0.0928 - learning_ra


✅ Model_C entrenado!
   Epochs ejecutados: 263
   Mejor epoch: 233
   Mejor val_loss: 0.027467
   Loss final: 0.047492
   Val_loss final: 0.027513

🎉 TODOS LOS MODELOS ENTRENADOS


## Paso 6: Generar Embeddings por Escenario

**CLAVE PARA COHERENCIA ESPACIAL**: Extraemos embeddings para cada píxel en cada escenario por separado.

Esto nos permite:
1. Promediar embeddings de los 3 modelos para cada píxel
2. Hacer clustering por escenario
3. Calcular índice de resiliencia por escenario
4. Comparar cómo cambia la zonificación entre SSP245/370/585

In [23]:
print("="*80)
print("GENERANDO EMBEDDINGS POR ESCENARIO")
print("="*80)

# Diccionario para guardar embeddings por escenario
embeddings_by_scenario = {
    'ssp245': {},
    'ssp370': {},
    'ssp585': {}
}

# Extraer embeddings de cada modelo para cada escenario
for model_name, encoder in encoders.items():
    latent_dim = MODELS_CONFIG[model_name]['latent_dim']
    print(f"\n🧠 {model_name} (latent_dim={latent_dim}):")
    
    # Generar embeddings para cada escenario
    emb_245 = encoder.predict(X_ssp245, verbose=0)
    emb_370 = encoder.predict(X_ssp370, verbose=0)
    emb_585 = encoder.predict(X_ssp585, verbose=0)
    
    embeddings_by_scenario['ssp245'][model_name] = emb_245
    embeddings_by_scenario['ssp370'][model_name] = emb_370
    embeddings_by_scenario['ssp585'][model_name] = emb_585
    
    print(f"   SSP245: {emb_245.shape}")
    print(f"   SSP370: {emb_370.shape}")
    print(f"   SSP585: {emb_585.shape}")

print("\n✅ Embeddings generados para todos los modelos y escenarios")
print("="*80)

GENERANDO EMBEDDINGS POR ESCENARIO

🧠 Model_A (latent_dim=7):
   SSP245: (661, 7)
   SSP370: (661, 7)
   SSP585: (661, 7)

🧠 Model_B (latent_dim=12):
   SSP245: (661, 12)
   SSP370: (661, 12)
   SSP585: (661, 12)

🧠 Model_C (latent_dim=15):
   SSP245: (661, 15)
   SSP370: (661, 15)
   SSP585: (661, 15)

✅ Embeddings generados para todos los modelos y escenarios


In [24]:
print("="*80)
print("PROMEDIAR EMBEDDINGS DEL MULTIMODELO POR ESCENARIO")
print("="*80)

# Promediar embeddings de los 3 modelos para cada escenario
embeddings_avg = {}

for scenario in SCENARIOS:
    print(f"\n📊 {scenario.upper()}:")
    
    # Obtener embeddings de los 3 modelos
    emb_A = embeddings_by_scenario[scenario]['Model_A']  # (661, 7)
    emb_B = embeddings_by_scenario[scenario]['Model_B']  # (661, 12)
    emb_C = embeddings_by_scenario[scenario]['Model_C']  # (661, 15)
    
    # Concatenar horizontalmente y promediar
    # Opción 1: Concatenar (661, 7+12+15=34)
    emb_concat = np.concatenate([emb_A, emb_B, emb_C], axis=1)
    
    embeddings_avg[scenario] = emb_concat
    
    print(f"   Model_A: {emb_A.shape}")
    print(f"   Model_B: {emb_B.shape}")
    print(f"   Model_C: {emb_C.shape}")
    print(f"   ✅ Concatenado: {emb_concat.shape}")

print("\n" + "="*80)
print("🎯 EMBEDDINGS MULTIMODELO LISTOS PARA CLUSTERING")
print("   Cada escenario: (661 píxeles, 34 features latentes)")
print("="*80)

PROMEDIAR EMBEDDINGS DEL MULTIMODELO POR ESCENARIO

📊 SSP245:
   Model_A: (661, 7)
   Model_B: (661, 12)
   Model_C: (661, 15)
   ✅ Concatenado: (661, 34)

📊 SSP370:
   Model_A: (661, 7)
   Model_B: (661, 12)
   Model_C: (661, 15)
   ✅ Concatenado: (661, 34)

📊 SSP585:
   Model_A: (661, 7)
   Model_B: (661, 12)
   Model_C: (661, 15)
   ✅ Concatenado: (661, 34)

🎯 EMBEDDINGS MULTIMODELO LISTOS PARA CLUSTERING
   Cada escenario: (661 píxeles, 34 features latentes)


## Paso 7: Clustering K-means por Escenario

Aplicamos K-means sobre los embeddings concatenados de cada escenario para identificar zonas climáticas homogéneas.

**Objetivo**: Identificar patrones espaciales en los embeddings que reflejen zonificación geográfica coherente.

In [27]:
print("="*80)
print("CLUSTERING K-MEANS POR ESCENARIO")
print("="*80)

# Probar diferentes valores de K y seleccionar el óptimo por silhouette score
K_RANGE = range(3, 11)  # Probar de 3 a 10 clusters

clustering_results = {}

for scenario in SCENARIOS:
    print(f"\n{'='*80}")
    print(f"🎯 CLUSTERING: {scenario.upper()}")
    print(f"{'='*80}")
    
    X_emb = embeddings_avg[scenario]  # (661, 34)
    
    silhouette_scores = []
    kmeans_models = []
    
    print(f"\n📊 Probando diferentes valores de K...")
    for k in K_RANGE:
        kmeans = KMeans(
            n_clusters=k,
            init='k-means++',
            n_init=20,
            max_iter=500,
            random_state=42
        )
        labels = kmeans.fit_predict(X_emb)
        score = silhouette_score(X_emb, labels)
        
        silhouette_scores.append(score)
        kmeans_models.append(kmeans)
        
        print(f"   K={k}: silhouette={score:.4f}")
    
    # Seleccionar K óptimo (máximo silhouette)
    best_idx = np.argmax(silhouette_scores)
    best_k = list(K_RANGE)[best_idx]
    best_score = silhouette_scores[best_idx]
    best_kmeans = kmeans_models[best_idx]
    best_labels = best_kmeans.labels_
    
    print(f"\n✅ K óptimo seleccionado: {best_k}")
    print(f"   Silhouette score: {best_score:.4f}")
    print(f"   Distribución de clusters:")
    unique, counts = np.unique(best_labels, return_counts=True)
    for cluster_id, count in zip(unique, counts):
        print(f"      Cluster {cluster_id}: {count} píxeles ({count/661*100:.1f}%)")
    
    # Guardar resultados
    clustering_results[scenario] = {
        'kmeans': best_kmeans,
        'labels': best_labels,
        'k': best_k,
        'silhouette': best_score,
        'all_scores': silhouette_scores,
        'embeddings': X_emb
    }

print(f"\n{'='*80}")
print("🎉 CLUSTERING COMPLETADO PARA TODOS LOS ESCENARIOS")
print(f"{'='*80}")

CLUSTERING K-MEANS POR ESCENARIO

🎯 CLUSTERING: SSP245

📊 Probando diferentes valores de K...
   K=3: silhouette=0.2961
   K=4: silhouette=0.2966
   K=5: silhouette=0.2902
   K=6: silhouette=0.2873
   K=7: silhouette=0.2954
   K=8: silhouette=0.2953
   K=9: silhouette=0.2985
   K=10: silhouette=0.2970

✅ K óptimo seleccionado: 9
   Silhouette score: 0.2985
   Distribución de clusters:
      Cluster 0: 94 píxeles (14.2%)
      Cluster 1: 63 píxeles (9.5%)
      Cluster 2: 98 píxeles (14.8%)
      Cluster 3: 36 píxeles (5.4%)
      Cluster 4: 89 píxeles (13.5%)
      Cluster 5: 52 píxeles (7.9%)
      Cluster 6: 65 píxeles (9.8%)
      Cluster 7: 57 píxeles (8.6%)
      Cluster 8: 107 píxeles (16.2%)

🎯 CLUSTERING: SSP370

📊 Probando diferentes valores de K...
   K=3: silhouette=0.2914
   K=4: silhouette=0.2720
   K=5: silhouette=0.2687
   K=6: silhouette=0.2932
   K=7: silhouette=0.3110
   K=8: silhouette=0.3106
   K=9: silhouette=0.3063
   K=10: silhouette=0.3138

✅ K óptimo selecciona

## Paso 8: Índice de Resiliencia

Calculamos el índice de resiliencia basado en la variabilidad de los embeddings. 

**Concepto**: Píxeles con menor variabilidad en su representación latente son más resilientes (estables) ante diferentes escenarios climáticos.

Usamos la distancia al centroide del cluster como métrica de resiliencia invertida.

In [26]:
print("="*80)
print("CÁLCULO DEL ÍNDICE DE RESILIENCIA POR ESCENARIO")
print("="*80)

resilience_results = {}

for scenario in SCENARIOS:
    print(f"\n{'='*80}")
    print(f"🎯 RESILIENCIA: {scenario.upper()}")
    print(f"{'='*80}")
    
    # Obtener datos del clustering
    kmeans = clustering_results[scenario]['kmeans']
    labels = clustering_results[scenario]['labels']
    X_emb = clustering_results[scenario]['embeddings']
    centroids = kmeans.cluster_centers_
    
    # Calcular distancia de cada píxel a su centroide
    distances = np.zeros(len(X_emb))
    for i in range(len(X_emb)):
        cluster_id = labels[i]
        centroid = centroids[cluster_id]
        distances[i] = np.linalg.norm(X_emb[i] - centroid)
    
    # Normalizar distancias a rango [0, 1]
    distances_norm = (distances - distances.min()) / (distances.max() - distances.min())
    
    # Índice de resiliencia: 1 - distancia normalizada
    # (Mayor resiliencia = menor distancia al centroide)
    resilience_index = 1 - distances_norm
    
    # Estadísticos
    print(f"\n📊 Estadísticos de resiliencia:")
    print(f"   Media: {resilience_index.mean():.4f}")
    print(f"   Std: {resilience_index.std():.4f}")
    print(f"   Min: {resilience_index.min():.4f}")
    print(f"   Max: {resilience_index.max():.4f}")
    print(f"   Mediana: {np.median(resilience_index):.4f}")
    
    # Top 10 píxeles más resilientes
    top10_idx = np.argsort(resilience_index)[-10:][::-1]
    print(f"\n🏆 Top 10 píxeles más resilientes:")
    for rank, idx in enumerate(top10_idx, 1):
        print(f"   {rank}. Píxel {idx}: resilience={resilience_index[idx]:.4f}, cluster={labels[idx]}")
    
    # Guardar resultados
    resilience_results[scenario] = {
        'resilience_index': resilience_index,
        'distances': distances,
        'distances_norm': distances_norm,
        'labels': labels
    }

print(f"\n{'='*80}")
print("✅ ÍNDICE DE RESILIENCIA CALCULADO PARA TODOS LOS ESCENARIOS")
print(f"{'='*80}")

CÁLCULO DEL ÍNDICE DE RESILIENCIA POR ESCENARIO

🎯 RESILIENCIA: SSP245

📊 Estadísticos de resiliencia:
   Media: 0.6713
   Std: 0.1575
   Min: 0.0000
   Max: 1.0000
   Mediana: 0.6878

🏆 Top 10 píxeles más resilientes:
   1. Píxel 119: resilience=1.0000, cluster=5
   2. Píxel 223: resilience=0.9996, cluster=1
   3. Píxel 429: resilience=0.9862, cluster=1
   4. Píxel 141: resilience=0.9844, cluster=8
   5. Píxel 362: resilience=0.9838, cluster=8
   6. Píxel 306: resilience=0.9834, cluster=8
   7. Píxel 135: resilience=0.9668, cluster=7
   8. Píxel 537: resilience=0.9602, cluster=1
   9. Píxel 363: resilience=0.9583, cluster=4
   10. Píxel 462: resilience=0.9481, cluster=8

🎯 RESILIENCIA: SSP370

📊 Estadísticos de resiliencia:
   Media: 0.6244
   Std: 0.1777
   Min: 0.0000
   Max: 1.0000
   Mediana: 0.6364

🏆 Top 10 píxeles más resilientes:
   1. Píxel 484: resilience=1.0000, cluster=9
   2. Píxel 488: resilience=0.9974, cluster=5
   3. Píxel 438: resilience=0.9742, cluster=2
   4. Píxel

## Paso 9: Validación de Coherencia Espacial - Moran's I

**MOMENTO CRÍTICO**: Calculamos el índice de Moran's I para validar la autocorrelación espacial del índice de resiliencia.

- **Moran's I > 0.3**: Indica autocorrelación espacial positiva (píxeles vecinos tienen valores similares) ✅
- **Moran's I ≈ 0**: No hay autocorrelación espacial ❌
- **Moran's I < 0**: Autocorrelación negativa (patrón tipo tablero de ajedrez) ❌

Este es el test definitivo para verificar que logramos coherencia geográfica.

In [28]:
def calculate_morans_i(values, mask, grid_shape):
    """
    Calcula el índice de Moran's I para autocorrelación espacial
    
    Args:
        values: Array 1D con valores para píxeles válidos (661,)
        mask: Boolean array 2D indicando píxeles válidos (lat, lon)
        grid_shape: Tupla con dimensiones del grid (n_lat, n_lon)
    
    Returns:
        morans_i: Valor del índice de Moran's I
    """
    n_lat, n_lon = grid_shape
    
    # Crear grid 2D con valores (rellenar con NaN donde mask=False)
    grid_2d = np.full(grid_shape, np.nan)
    grid_2d[mask] = values
    
    # Listas para acumular productos
    numerator_sum = 0
    denominator_sum = 0
    weight_sum = 0
    
    mean_val = np.nanmean(grid_2d)
    n_valid = 0
    
    # Iterar sobre cada píxel válido
    for i in range(n_lat):
        for j in range(n_lon):
            if not mask[i, j]:
                continue
            
            val_i = grid_2d[i, j]
            n_valid += 1
            
            # Vecinos (4-conectividad: arriba, abajo, izq, der)
            neighbors = []
            if i > 0 and mask[i-1, j]:  # arriba
                neighbors.append(grid_2d[i-1, j])
            if i < n_lat-1 and mask[i+1, j]:  # abajo
                neighbors.append(grid_2d[i+1, j])
            if j > 0 and mask[i, j-1]:  # izquierda
                neighbors.append(grid_2d[i, j-1])
            if j < n_lon-1 and mask[i, j+1]:  # derecha
                neighbors.append(grid_2d[i, j+1])
            
            # Acumular productos con vecinos
            for val_j in neighbors:
                numerator_sum += (val_i - mean_val) * (val_j - mean_val)
                weight_sum += 1
            
            # Acumular para denominador
            denominator_sum += (val_i - mean_val) ** 2
    
    # Calcular Moran's I
    if weight_sum > 0 and denominator_sum > 0:
        morans_i = (n_valid / weight_sum) * (numerator_sum / denominator_sum)
    else:
        morans_i = 0.0
    
    return morans_i

print("✅ Función calculate_morans_i definida")

✅ Función calculate_morans_i definida


In [30]:
print("="*80)
print("VALIDACIÓN: MORAN'S I - AUTOCORRELACIÓN ESPACIAL")
print("="*80)

morans_results = {}

for scenario in SCENARIOS:
    print(f"\n{'='*80}")
    print(f"🎯 MORAN'S I: {scenario.upper()}")
    print(f"{'='*80}")
    
    resilience_index = resilience_results[scenario]['resilience_index']
    
    # Calcular Moran's I
    morans_i = calculate_morans_i(resilience_index, MASK, grid_shape)
    
    print(f"\n📊 Índice de Moran's I: {morans_i:.4f}")
    
    # Interpretación
    if morans_i > 0.3:
        interpretation = "✅ EXCELENTE - Fuerte autocorrelación espacial positiva"
        status = "ÉXITO"
    elif morans_i > 0.1:
        interpretation = "✓ BUENO - Autocorrelación espacial moderada"
        status = "ACEPTABLE"
    elif morans_i > 0:
        interpretation = "⚠️  DÉBIL - Autocorrelación espacial leve"
        status = "MEJORABLE"
    elif morans_i < -0.1:
        interpretation = "❌ MALO - Autocorrelación negativa (patrón aleatorio)"
        status = "FALLO"
    else:
        interpretation = "⚠️  NULO - Sin autocorrelación espacial"
        status = "FALLO"
    
    print(f"   {interpretation}")
    print(f"   Status: {status}")
    
    morans_results[scenario] = {
        'morans_i': morans_i,
        'interpretation': interpretation,
        'status': status
    }

# Resumen final
print(f"\n{'='*80}")
print("🎯 RESUMEN DE VALIDACIÓN ESPACIAL")
print(f"{'='*80}")
for scenario in SCENARIOS:
    mi = morans_results[scenario]['morans_i']
    status = morans_results[scenario]['status']
    print(f"   {scenario.upper()}: Moran's I = {mi:.4f} ({status})")

print(f"\n{'='*80}")

VALIDACIÓN: MORAN'S I - AUTOCORRELACIÓN ESPACIAL

🎯 MORAN'S I: SSP245

📊 Índice de Moran's I: -0.0144
   ⚠️  NULO - Sin autocorrelación espacial
   Status: FALLO

🎯 MORAN'S I: SSP370

📊 Índice de Moran's I: 0.0008
   ⚠️  DÉBIL - Autocorrelación espacial leve
   Status: MEJORABLE

🎯 MORAN'S I: SSP585

📊 Índice de Moran's I: -0.0051
   ⚠️  NULO - Sin autocorrelación espacial
   Status: FALLO

🎯 RESUMEN DE VALIDACIÓN ESPACIAL
   SSP245: Moran's I = -0.0144 (FALLO)
   SSP370: Moran's I = 0.0008 (MEJORABLE)
   SSP585: Moran's I = -0.0051 (FALLO)

